# NutAssembly (Robosuite) — Interactive Notebook

Create a Robosuite `NutAssembly` environment (e.g., `NutAssemblySquare`), compute a handle target from the nut quaternion, run a simple scripted primitive to line up/grasp the handle, and record an inline video.

This is a best-effort notebook — adapt camera names, controller config, and action indices to your local setup.

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import io
import base64
import cv2
import numpy as np
import imageio
from IPython.display import HTML, display
from scipy.spatial.transform import Rotation as R
from hires_vic.wrappers import WipeMetricWrapper, GeometricWrapper, FixedGripperWrapper

from hires_vic.envs.riemannian_controller import RiemannianController
import robosuite.controllers.parts.controller_factory as factory
factory.arm_controllers.OperationalSpaceController = RiemannianController

import logging

# from robosuite.utils.log_utils import ROBOSUITE_DEFAULT_LOGGER

# Suppress all robosuite warnings (like joint limits and macro files)
# ROBOSUITE_DEFAULT_LOGGER.setLevel(logging.ERROR)

def display_video(path, width=640):
    mp4 = open(path,'rb').read()
    data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()
    html = f'<video width="{width}" controls><source src="{data_url}" type="video/mp4"></video>'
    display(HTML(html))

def frames_to_mp4(frames, path, fps=30):
    imageio.mimwrite(path, frames, fps=fps, macro_block_size=None)


[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)
/home/cjimenez/miniconda3/envs/tfm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Registering custom environments...


In [2]:
def capture_frame_from_env(env):
    try:
        # robosuite common call
        frame = env.render(camera_name='frontview')
        if frame is None:
            frame = env.render()
    except Exception:
        frame = None
    if frame is None:
        try:
            frame = env.unwrapped.sim.render(width=640, height=480, camera_name='frontview')
        except Exception:
            frame = None
    return frame

In [3]:
import numpy as np
import gymnasium as gym
global_frames = []
class RobosuiteTeleportWrapper(gym.Wrapper):
    """
    Bypasses the pick phase! Teleports the nut directly into the robot's 
    hand and settles the physics to start the episode ready for insertion.
    """
    def __init__(self, env, setup_steps=140):
        super().__init__(env)
        self.setup_steps = setup_steps
        print('init teleport wrapper')

    def reset(self, **kwargs):
        obs = self.env.reset(**kwargs)
        frame = capture_frame_from_env(self.env)
        frame = np.flipud(frame)
        frame = np.ascontiguousarray(frame)
        
        # Print "Setup Phase" on the video so you know it's your dummy loop
        text = f"Setup: 0  - Teleporting nut..."
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(frame, text, (20, 40), font, 1.0, (0,0,0), 4, cv2.LINE_AA)
        cv2.putText(frame, text, (20, 40), font, 1.0, (0, 255, 0), 2, cv2.LINE_AA) # Green text!
        global_frames.append(frame)
        # In Robosuite, the MuJoCo simulation state lives here:
        sim = self.env.unwrapped.sim

        # 2. Get the TCP (Gripper) Pose
        # Robosuite caches this in the unwrapped environment
        raw_obs = self.env.unwrapped._get_observations()
        eef_pos = raw_obs['robot0_eef_pos']
        eef_quat = raw_obs['robot0_eef_quat'] # [x, y, z, w]
        # 3. Calculate where you want the Nut to be
        # (e.g., perfectly centered between the fingers)
        nut_target_pos = eef_pos + np.array([0.048, 0.0, 0.015]) # Small Z offset so it sits in the pads

        r_eef = R.from_quat(eef_quat)
        r_flip = R.from_euler('z', np.pi)
        r_nut = r_eef * r_flip
        flipped_quat = r_nut.as_quat() # Returns standard [x, y, z, w]

        # mujoco expects [w, x, y, z] 
        nut_target_quat_mujoco = np.array([flipped_quat[3], flipped_quat[0], flipped_quat[1], flipped_quat[2]])
        
        # 4. Generate Randomized Peg Hover Target
        try:
            # print(type(self.env.unwrapped))
            peg_key = 'peg1' if 'square' in type(self.env.unwrapped).__name__.lower() else 'peg2'
            print(f"Identified peg key: {peg_key}")
            peg_id = sim.model.body_name2id(peg_key)
            peg_base_pos = np.array(sim.data.body_xpos[peg_id])
            
            # Curriculum Noise: +/- 2.0cm offset so the RL agent is forced to search!
            noise_x = np.random.uniform(-0.020, 0.020)
            noise_y = np.random.uniform(-0.020, 0.020)
            
            # Hover ~12cm above the base of the peg
            hover_target = peg_base_pos + np.array([noise_x-0.048, noise_y, 0.12])
        except Exception as e:
            # Safe fallback just in case the peg name changes
            print("Peg hover target generation failed! Check the peg body name.", e)
            hover_target = eef_pos 

        action_dim = self.env.action_space.shape[0] # Expects raw 7D array
        
        
        for dummy_t in range(self.setup_steps):
            scripted_action = np.zeros(action_dim, dtype=np.float32)
            # Recalculate the arm's position every frame
            current_eef = self.env.unwrapped._get_observations()['robot0_eef_pos']
            gain = 10.0
            mid_target = hover_target + np.array([0.0, 0.0, 0.04]) 
            # mid_target = np.array([0.0, 0.0, 1]) 

            if dummy_t < self.setup_steps // 2.75:
                pos_error = (mid_target - current_eef)
                # pos_error = (np.array([0.19180354, -0.30, 1.07]) - current_eef)
            else:
                # print('Switching to final hover target!')
                pos_error = (hover_target - current_eef)
                # pos_error = (np.array([0.19180354, -0.30, 0.97]) - current_eef)
            
            pos_error *= gain * 0.05 # Scale down for stability
            
            if action_dim < 16:
                pos_idx = 6
                scripted_action[0:pos_idx] = np.array([-0.89, -0.89, -0.52,  -0.8, -0.5, -0.5]) 
                # 
            else:
                pos_idx = 9
                scripted_action[0:pos_idx] = np.array([0.0, -0.0, 0.5, 0, 0, 0, -0.8, -0.5, -0.5]) 
                # scripted_action[0:pos_idx] = np.array([1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0])
            scripted_action[pos_idx:pos_idx+3] = np.clip(pos_error, -1.0, 1.0)
            scripted_action[-1] = 1.0 
            # print(f"Dummy Step {dummy_t+1}/{self.setup_steps}, Action ({len(scripted_action)}): {scripted_action[pos_idx:pos_idx+3]}")
            print(f'ceef: {current_eef} , err: {pos_error} , t: {hover_target}')
            obs, reward, terminated, truncated, info = self.env.step(scripted_action)
            
            # moving the nut after the gripper starts closing
            # seems to help it settle better in the hand before flight
            if dummy_t == 2:
                try:
                    joint_name = "SquareNut_joint0" if 'square' in type(self.env.unwrapped).__name__.lower() else "RoundNut_joint0"
                    
                    # Find exactly where in the giant array the nut lives
                    qpos_addr = sim.model.get_joint_qpos_addr(joint_name)
                    qvel_addr = sim.model.get_joint_qvel_addr(joint_name)
                    
                    # Teleport: Overwrite the 7 positional values (X, Y, Z, Qw, Qx, Qy, Qz)
                    sim.data.qpos[qpos_addr[0] : qpos_addr[1]] = np.concatenate([nut_target_pos, nut_target_quat_mujoco])
                    
                    # Kill momentum: Overwrite the 6 velocity values (Linear X/Y/Z, Angular X/Y/Z) to zero
                    sim.data.qvel[qvel_addr[0] : qvel_addr[1]] = np.zeros(6)
                    
                    # Tell MuJoCo to apply these hardcoded changes immediately!
                    sim.forward()

                    # print(f"EEF: {current_eef} , Peg pose {peg_base_pos} with mid target {mid_target} and hover {hover_target}")

                except Exception as e:
                    print("Teleport failed! Check the joint name of the object.", e)


            try:
                frame = capture_frame_from_env(self.env)
                if frame is not None:
                    if frame.ndim == 4:
                        frame = frame[0]
                    frame = np.flipud(frame)
                    frame = np.ascontiguousarray(frame)
                    
                    # Updated overlay text so you know it is flying
                    text = f"Setup: {dummy_t}  (Flying to Peg!)"
                    font = cv2.FONT_HERSHEY_SIMPLEX
                    cv2.putText(frame, text, (20, 40), font, 1.0, (0,0,0), 4, cv2.LINE_AA)
                    cv2.putText(frame, text, (20, 40), font, 1.0, (0, 255, 0), 2, cv2.LINE_AA)
                    
                    global_frames.append(frame)
            except Exception as e:
                print(f"Frame capture failed during setup step {dummy_t}: {e}")
                pass

        return obs

In [4]:
# Create a NutAssembly env (best-effort). Change env_name if needed.
env_name = 'NutAssemblySquare'
fixed_kp = 120
use_spd_manifold = False
use_lie_group = False
use_llm_prior = False
use_fixed = False
try:
    import robosuite as suite
    from robosuite.wrappers import GymWrapper
    from robosuite import load_composite_controller_config
    
    controller_config = load_composite_controller_config(controller="BASIC", robot="panda")
    phantom_parts = ["left", "torso", "head", "base", "legs"]
    for part in phantom_parts:
        controller_config["body_parts"].pop(part, None)
    arm_config = controller_config["body_parts"]["right"]
    arm_config["type"] = "OSC_POSE"
    arm_config["impedance_mode"] = "riemannian_kp" if use_spd_manifold else "fixed" if use_fixed else "variable_kp"
    arm_config["kp_limits"] = [1, 300]
    arm_config["damping_ratio_limits"] = [1.0, 1.0]
    if use_fixed:
        arm_config["kp"] = fixed_kp
    env = suite.make(
        env_name=env_name,
        robots='Panda',
        controller_configs=controller_config, 
        has_renderer=False, 
        use_object_obs=True, 
        has_offscreen_renderer=True, 
        use_camera_obs=True, 
        camera_names='frontview', 
        reward_shaping=True
    )
    env = GymWrapper(env)
    env = GeometricWrapper(
        env, 
        use_spd_manifold=use_spd_manifold, 
        use_lie_group=use_lie_group, 
        use_llm_prior=use_llm_prior,
        use_fixed=use_fixed,
        is_eval=True
    )
    env = RobosuiteTeleportWrapper(env)
    env = FixedGripperWrapper(env)
    
    print('Wrapped with GeometricWrapper')
except Exception as e:
    raise RuntimeError('Failed to create Robosuite NutAssembly environment. Ensure robosuite is installed and the env name is correct.') from e

print('Env created:', env_name, 'wrapped type:', type(env))
# obs = env.reset()
# print('Reset done. If obs is dict, keys: ', list(obs.keys()) if isinstance(obs, dict) else type(obs))


[robosuite INFO] Loading controller configuration from: /home/cjimenez/miniconda3/envs/tfm/lib/python3.12/site-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)


/home/cjimenez/miniconda3/envs/tfm/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/cjimenez/miniconda3/envs/tfm/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(



            🔧 Robosuite Wrapper Initialized
            | SPD: False
            | Lie Group: False
            | Diag Manifold: False
            | Fixed: False
        
▶️ Action space strictly set to [-1, 1] with shape: (13,)
▶️ Observation space shape after flattening:  (64,)
init teleport wrapper
▶️ New action space with Fixed gripper: (12,)
Wrapped with GeometricWrapper
Env created: NutAssemblySquare wrapped type: <class 'hires_vic.wrappers.fixed_gripper.FixedGripperWrapper'>


In [5]:
def compute_handle_pos(raw_obs, offset=0.04, quat_debug=False):
    "Return (handle_pos, nut_pos, chosen_quat) or (None, None, None)"
    if not isinstance(raw_obs, dict):
        return None, None, None
    nut_pos = None
    nut_quat = None
    for k, v in raw_obs.items():
        kl = k.lower()
        if 'nut' in kl and 'pos' in kl and 'to_' not in kl:
            nut_pos = np.asarray(v).flatten()[:3]
        if 'nut' in kl and 'quat' in kl and 'to_' not in kl:
            nut_quat = np.asarray(v).flatten()[:4]
    if nut_pos is None:
        return None, None, None
    if nut_quat is None:
        return nut_pos, nut_pos, None
    q_raw = np.asarray(nut_quat).flatten()
    if q_raw.size < 4:
        return nut_pos, nut_pos, None
    # try two common orderings and pick the one whose local Z aligns best with world Z (upright heuristic)
    orders = [q_raw[:4], np.array([q_raw[1], q_raw[2], q_raw[3], q_raw[0]])]
    best = None
    best_score = -1.0
    best_rot = None
    for q in orders:
        try:
            nq = q / (np.linalg.norm(q) + 1e-12)
            rot = R.from_quat(nq)
            local_z = rot.apply([0.0, 0.0, 1.0])
            score = abs(np.dot(local_z, np.array([0.0, 0.0, 1.0])))
            if score > best_score:
                best_score = score
                best = nq
                best_rot = rot
        except Exception:
            continue
    if best_rot is None:
        return nut_pos, nut_pos, nut_quat
    # use local +X, enforce negative sign and offset (nut_pos - local_x * offset)
    offset_v = best_rot.apply([offset, 0.0, 0.0])
    # axis_unit = local_x / (np.linalg.norm(local_x) + 1e-12)
    handle_pos = nut_pos + offset_v
    if quat_debug:
        print('Chosen quat (x,y,z,w):', best, 'offset_x=', offset_v, 'best_rot=', best_rot, 'nut_quat=', nut_quat)
    return handle_pos, nut_pos, best


In [6]:
def determine_action_indices(env):
    action_dim = int(env.action_space.shape[-1])
    # print(action_dim, 'action space shape:', env.action_space.shape)
    # Heuristic: if action_dim large, assume SPD manifold layout -> pos_idx 9, else 6
    if hasattr(env, 'use_spd_manifold') and getattr(env, 'use_spd_manifold'):
        pos_idx = 9
    else:
        pos_idx = 6 if action_dim > 7 else 0 # fixed
    gripper_idx = max(0, action_dim - 1)
    return action_dim, pos_idx, gripper_idx

def scripted_primitive_policy(env, raw_obs, prev_delta_ori, step_phase=0.0, quat_debug=False):
    "Scripted primitive policy for NutAssembly: uses `compute_handle_pos` and a simple PD approach."
    action_dim, pos_idx, gripper_idx = determine_action_indices(env)
    action = np.zeros((action_dim,), dtype=np.float32)

    # Extract TCP pose/quaternion from raw_obs when available
    tcp_pos = None
    tcp_quat = None
    try:
        if isinstance(raw_obs, dict):
            if 'robot0_eef_pos' in raw_obs:
                tcp_pos = np.asarray(raw_obs['robot0_eef_pos']).flatten()[:3]
            if 'robot0_eef_quat' in raw_obs:
                tcp_quat = np.asarray(raw_obs['robot0_eef_quat']).flatten()[:4]
    except Exception:
        pass

    handle_pos, nut_pos, chosen_quat = compute_handle_pos(raw_obs, offset=0.04, quat_debug=quat_debug)
    if handle_pos is None:
        # Fallback: sample an action or return zeros
        try:
            return env.action_space.sample()
        except Exception:
            return action

    # Obtain peg position (MuJoCo direct) as a fallback target
    peg_pos = None
    try:
        sim = env.unwrapped.sim
        peg_key = 'peg1' if  'Square' in env_name else 'peg2'
        peg_id = sim.model.body_name2id(peg_key)
        peg_pos = np.array(sim.data.body_xpos[peg_id]) + np.array([0.0, 0.0, 0.08])
    except Exception:
        peg_pos = nut_pos

    # print('handle_pos:', handle_pos, 'nut_pos:', nut_pos, 'peg_pos:', peg_pos)

    # print(env)
    setattr(env, 'suppress_forced_gripper', True)
    # print('before getattr:', getattr(env, 'suppress_forced_gripper', False))

    # Targets: grasp (handle), midpoint, hover over peg
    grasp_target = handle_pos + np.array([0.0, 0.0, 0.05])
    mid_target1 = handle_pos + (peg_pos - handle_pos) * 0.25
    mid_target2 = handle_pos + (peg_pos - handle_pos) * 0.5
    mid_target3 = handle_pos + (peg_pos - handle_pos) * 0.75
    hover_target = peg_pos + np.array([0.0, 0.0, 0.01])

    OPEN = -1.0
    CLOSE = 1.0

    # Phase selection based on normalized step_phase in [0,1]
    phase = float(step_phase) if step_phase is not None else 0.0
    if phase < 0.30:
        target = handle_pos + np.array([0.0, 0.0, 0.1])
        gripper_act = OPEN
    elif phase < 0.50:
        target = handle_pos + np.array([0.0, 0.0, 0.0])
        gripper_act = OPEN
    elif phase < 0.60:
        target = handle_pos + np.array([0.0, 0.0, 0.0])
        gripper_act = CLOSE
    elif phase < 0.70:
        target = handle_pos + np.array([0.0, 0.0, 0.1])
        gripper_act = CLOSE
    elif phase < 0.85:
        target = mid_target2 + np.array([0.0, 0.0, 0.1])
        gripper_act = CLOSE
    else:
        target = hover_target
        gripper_act = CLOSE

    print(target, tcp_pos)

    # Position PD (simple proportional for demo). Tune gain as needed.
    if tcp_pos is None:
        delta_pos = np.zeros(3, dtype=np.float32)
    else:
        primitive_approach_gain = 10
        delta_pos = (target - tcp_pos) * (primitive_approach_gain * 0.01)

    # Orientation correction: small rotation-vector in EEF local frame
    delta_ori = np.zeros(3, dtype=np.float32)
    ori_scale = 0.1       # The Rotational Gain (Smaller fractional steps)
    smooth_alpha = 0.15   # The Acceleration Curve (Gentle ease-in)
    max_ori_step = 0.01
    
    if tcp_quat is not None and chosen_quat is not None:
        try:
            r_current = R.from_quat(tcp_quat)
            r_nut = R.from_quat(chosen_quat)
            
            target_z = np.array([0.0, 0.0, -1.0])
            
            # --- THE FIX: Phase-Dependent Orientation ---
            if phase < 0.60:
                # 1. PRE-GRASP: Actively track and align to the Nut
                nut_x = r_nut.apply([1.0, 0.0, 0.0])
                target_x_base = np.array([nut_x[0], nut_x[1], 0.0])
                target_x_base = target_x_base / (np.linalg.norm(target_x_base) + 1e-12)
                
                gripper_x = r_current.apply([1.0, 0.0, 0.0])
                gripper_x_flat = np.array([gripper_x[0], gripper_x[1], 0.0])
                gripper_x_flat = gripper_x_flat / (np.linalg.norm(gripper_x_flat) + 1e-12)
                
                # The 180-degree symmetry check
                if np.dot(target_x_base, gripper_x_flat) < 0:
                    target_x = -target_x_base
                else:
                    target_x = target_x_base
            else:
                # 2. TRANSPORT: Stop tracking the nut! 
                # Use the gripper's OWN current flattened X-axis as the target.
                # This mathematically forces the Twist (Yaw) error to be 0.0, 
                # so the controller only fights to keep the wrist vertically leveled.
                gripper_x = r_current.apply([1.0, 0.0, 0.0])
                target_x = np.array([gripper_x[0], gripper_x[1], 0.0])
                target_x = target_x / (np.linalg.norm(target_x) + 1e-12)
                
            # 3. Build the single, perfectly stable target matrix
            target_y = np.cross(target_z, target_x)
            target_matrix = np.column_stack((target_x, target_y, target_z))
            r_target = R.from_matrix(target_matrix)
            
            # Calculate exact World Frame error
            r_error_world = r_target * r_current.inv()
            delta_ori_raw = r_error_world.as_rotvec()
            
            # Scale, smooth, and CAP THE SPEED
            scaled = delta_ori_raw * ori_scale
            smoothed = prev_delta_ori * (1.0 - smooth_alpha) + scaled * smooth_alpha
            
            norm = np.linalg.norm(smoothed)
            if norm > max_ori_step and norm > 1e-12:
                # print(smoothed, 'norm=', norm, 'exceeds max_ori_step=', max_ori_step, '- capping!')
                smoothed = (smoothed / norm) * max_ori_step
                
            delta_ori = smoothed
            prev_delta_ori = smoothed
            
        except Exception as e:
            print('Orientation compute failed:', e)
            delta_ori = np.zeros(3, dtype=np.float32)

    # delta_pos = np.zeros(3, dtype=np.float32)
    delta_pos = np.clip(delta_pos, -1.0, 1.0)
    delta_ori = np.clip(delta_ori, -1.0, 1.0)
    try:
        action[:pos_idx] = np.array([0.5, 0.5, 0.5, 0.2, 0.2, 0.2, 0.5, 0.5, 0.5]) 
        # action[:pos_idx] = np.array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5]) 
        action[pos_idx:pos_idx + 3] = delta_pos
        action[pos_idx + 3:pos_idx + 6] = delta_ori
        action[gripper_idx] = float(gripper_act)
    except Exception as e:
        # best-effort fallback
        print('Failed to insert action components, falling back to sampling or zeros.', e)
        try:
            action = env.action_space.sample()
        except Exception:
            pass

    return action, target, tcp_pos, nut_pos, prev_delta_ori

## TeleportWrapper example usage:

In [7]:
# Run the scripted primitive and record a short video
out_dir = 'outputs'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'nutassembly_primitive.mp4')
num_steps = 250
# frames = []

# 1. The Wrapper intercepts this. The nut is teleported and grasped before this line finishes!
obs = env.reset()

action_dim = int(env.action_space.shape[-1])
settle_action = np.zeros(action_dim, dtype=np.float32)

# 2. CRITICAL FIX: Keep the gripper CLOSED (+1.0) so it doesn't drop the teleported nut!
settle_action[-1] = 1.0 

# 3. Capture frames during the dummy/settling steps
for dummy_t in range(0):
    try:
        step_out = env.step(settle_action)
        # Keep tracking the observation so the next step has fresh data
        obs = step_out[0] if isinstance(step_out, (tuple, list)) else step_out
    except Exception:
        pass
        
    # --- Capture visual frame for dummy steps ---
    frame = capture_frame_from_env(env)
    if frame is None and isinstance(obs, dict):
        for k in ('frontview_image', 'frontview_camera', 'camera_front_image', 'frontview'):
            if k in obs:
                f = np.asarray(obs[k])
                if f is not None:
                    frame = f
                    break
    if frame is not None:
        if frame.ndim == 4:
            frame = frame[0]
        try:
            frame = np.flipud(frame)
            frame = np.ascontiguousarray(frame)
            
            # Print "Setup Phase" on the video so you know it's your dummy loop
            text = f"Setup: {dummy_t}  (Teleported!)"
            font = cv2.FONT_HERSHEY_SIMPLEX
            cv2.putText(frame, text, (20, 40), font, 1.0, (0,0,0), 4, cv2.LINE_AA)
            cv2.putText(frame, text, (20, 40), font, 1.0, (0, 255, 0), 2, cv2.LINE_AA) # Green text!
        except Exception:
            pass
        global_frames.append(frame)

if len(global_frames) == 0:
    print('No frames captured; enable offscreen rendering or adjust camera names.')
else:
    frames_to_mp4(global_frames, out_path, fps=30)
    print('Saved video to', out_path)
    # display_video(out_path)

Identified peg key: peg1
ceef: [-0.11025758  0.01077229  1.00061603] , err: [0.13691704 0.03855015 0.00469199] , t: [0.16357649 0.08787258 0.97      ]
ceef: [-0.1101461   0.01079139  1.0006331 ] , err: [0.13686129 0.0385406  0.00468345] , t: [0.16357649 0.08787258 0.97      ]
ceef: [-0.10980304  0.01085025  1.00068553] , err: [0.13668977 0.03851117 0.00465724] , t: [0.16357649 0.08787258 0.97      ]
ceef: [-0.10923009  0.01094879  1.00077263] , err: [0.13640329 0.0384619  0.00461369] , t: [0.16357649 0.08787258 0.97      ]
ceef: [-0.10844666  0.01108908  1.00087951] , err: [0.13601158 0.03839175 0.00456025] , t: [0.16357649 0.08787258 0.97      ]
ceef: [-0.10746427  0.01126546  1.00099524] , err: [0.13552038 0.03830356 0.00450238] , t: [0.16357649 0.08787258 0.97      ]
ceef: [-0.10628788  0.011481    1.00111735] , err: [0.13493219 0.03819579 0.00444132] , t: [0.16357649 0.08787258 0.97      ]
ceef: [-0.1049218   0.01173602  1.00124377] , err: [0.13424914 0.03806828 0.00437811] , t: [0

In [8]:
# # Run the scripted primitive and record a short video
# out_dir = 'outputs'
# os.makedirs(out_dir, exist_ok=True)
# out_path = os.path.join(out_dir, 'nutassembly_primitive.mp4')
# num_steps = 250
# frames = []
# obs = env.reset()
# action_dim = int(env.action_space.shape[-1])
# settle_action = np.zeros(action_dim, dtype=np.float32)
# settle_action[-1] = -1.0 # Robosuite OPEN (+1.0)

# for _ in range(20):
#     try:
#         step_out = env.step(settle_action)
#         # Keep tracking the observation so the next step has fresh data
#         obs = step_out[0] if isinstance(step_out, (tuple, list)) else step_out
#     except Exception:
#         pass

# prev_delta_ori = np.zeros(3, dtype=np.float32)
# for t in range(num_steps):
#     # Retrieve raw observations from the unwrapped env when possible
#     try:
#         raw_obs = env.unwrapped._get_observations()
#     except Exception:
#         raw_obs = obs if isinstance(obs, dict) else None
#     phase = float(t) / max(1, num_steps - 1)
#     action, target, tcp_pos, nut_pos, prev_delta_ori = scripted_primitive_policy(env, raw_obs, prev_delta_ori, step_phase=phase, quat_debug=False)
#     # Step the env with the scripted action
#     try:
#         obs, rew, terminated, truncated, info = env.step(action)
#     except Exception as e:
#         print('env.step failed:', e)
#         break
#     # Optionally print handle debug info occasionally
#     if t % 25 == 0:
#         try:
#             hpos, npos, q = compute_handle_pos(raw_obs)
#             print(f'step {t} handle_pos={hpos} nut_pos={npos} quat={q}')
#             # print(f'step {t}, phase {phase:.2f}: target={target} tcp_pos={tcp_pos} nut_pos={nut_pos}')
#         except Exception:
#             pass
#     # Capture a visual frame (with fallbacks)
#     frame = capture_frame_from_env(env)
#     if frame is None and isinstance(obs, dict):
#         for k in ('frontview_image', 'frontview_camera', 'camera_front_image', 'frontview'):
#             if k in obs:
#                 f = np.asarray(obs[k])
#                 if f is not None:
#                     frame = f
#                     break
#     if frame is not None:
#         # Some renderers produce upside-down frames; flip vertically if it looks tall
#         if frame.ndim == 4:
#             frame = frame[0]
#         try:
#             frame = np.flipud(frame)
#             # 2. Make sure the array is contiguous (required by OpenCV)
#             frame = np.ascontiguousarray(frame)
            
#             # 3. Add the Step text to the top-left corner
#             text = f"Step: {t}  Phase: {phase:.2f}"
#             font = cv2.FONT_HERSHEY_SIMPLEX
#             position = (20, 40) # (X, Y) coordinates from top-left
#             font_scale = 1.0
#             color = (255, 255, 255) # White text (R, G, B)
#             thickness = 2
            
#             # Draw a subtle black outline so it's readable on light backgrounds
#             cv2.putText(frame, text, position, font, font_scale, (0,0,0), thickness+2, cv2.LINE_AA)
#             # Draw the white text
#             cv2.putText(frame, text, position, font, font_scale, color, thickness, cv2.LINE_AA)
#         except Exception:
#             pass
#         frames.append(frame)
#     if terminated or truncated:
#         break

# setattr(env, 'suppress_forced_gripper', False)
# # print('after setattr:', getattr(env, 'suppress_forced_gripper', True))

# if len(frames) == 0:
#     print('No frames captured; enable offscreen rendering or adjust camera names.')
# else:
#     frames_to_mp4(frames, out_path, fps=30)
#     print('Saved video to', out_path)
#     display_video(out_path)

In [9]:
# nut_pos=[-0.1108856,  0.17959426, 0.82998947]
# nut_quat=[-8.76542435e-08,  5.48543713e-07,  1.57792594e-01,  9.87472277e-01]
# local_x = R.from_quat(nut_quat).apply([0.35, 0.0, 0.0])
# print('nut_pos=', nut_pos, 'nut_quat=', nut_quat, 'local_x=', local_x)
# # print('axis_unit=', axis_unit*0.5)
# handle_pos = nut_pos + local_x
# print('handle_pos=', handle_pos)

# -2.60821301e-02 -0.11166225